In [59]:
import numpy as np
import kagglehub
import os
import pandas as pd
from pathlib import Path
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter
import math
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Part 1: Scaled Dot-product attention

In [60]:
def softmax(x):
    x_max = np.max(x, axis=-1, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

# NumPy-only implementation
def scaled_dot_product_attention(Q, K, V):
    d = Q.shape[-1]
    scores = np.dot(Q, K.T) / np.sqrt(d)
    attention_weights = softmax(scores)
    output = np.dot(attention_weights, V)
    return output, attention_weights

## Part 2: Encoder-Decoder seq2seq with Scaled Dot Product Attention

Dataset: https://www.kaggle.com/datasets/shahadhamza/multi30k-dataset

has 4 components: train.en, train.fr, val.en, and val.fr


In [61]:
# Define the model

# Scaled Dot Product Attention as pytorch
class ScaledDotProductAttention(nn.Module):
    def forward(self, Q, K, V):
        d_k = Q.size(-1) ** 0.5
        scores = torch.bmm(Q, K.transpose(1, 2)) / d_k
        weights = torch.softmax(scores, dim=-1)
        return torch.bmm(weights, V)


# Tokenizer for vocabulary
class Vocabulary:
    PAD, SOS, EOS, UNK = 0, 1, 2, 3

    def __init__(self, max_size=10_000):
        self.max_size = max_size
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def build(self, sentences):
        counts = Counter(w for s in sentences for w in s.lower().split())
        for word, _ in counts.most_common(self.max_size - len(self.word2idx)):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, sentence, max_len):
        ids = [self.word2idx.get(w, self.UNK) for w in sentence.lower().split()]
        ids = [self.SOS] + ids[:max_len - 2]   # room for SOS + EOS
        ids.append(self.EOS)                    # always terminate with EOS
        ids += [self.PAD] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        return [self.idx2word.get(i, '<UNK>') for i in ids
                if i not in (self.PAD, self.SOS, self.EOS)]


# Dataset for the machine translation
class TranslationDataset(Dataset):
    def __init__(self, src_seqs, tgt_seqs):
        self.src = torch.tensor(src_seqs, dtype=torch.long)
        self.tgt = torch.tensor(tgt_seqs, dtype=torch.long)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]


# Encoder for Seq2Seq
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)
        outputs, (h, c) = self.lstm(emb)
        return outputs, h, c

# Decoder for Seq2Seq
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.attention = ScaledDotProductAttention()
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x, enc_outputs, h, c):
        emb = self.embedding(x)
        dec_out, (h, c) = self.lstm(emb, (h, c))
        context = self.attention(dec_out, enc_outputs, enc_outputs)
        combined = torch.cat([dec_out, context], dim=-1)
        logits = self.fc(combined)
        return logits, h, c

# Compiled Seq2Seq model
class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.encoder = enc
        self.decoder = dec

    def forward(self, src, tgt):
        enc_out, h, c = self.encoder(src)
        logits, _, _ = self.decoder(tgt, enc_out, h, c)
        return logits


In [62]:
# Hyper params for model
VOCAB_SIZE = 10_000
EMB_DIM = 256
HIDDEN_DIM = 512
MAX_LEN = 40
BATCH_SIZE = 64

# Load data
path = kagglehub.dataset_download('shahadhamza/multi30k-dataset')
train_en = Path(os.path.join(path, 'train.en')).read_text(encoding='utf-8').splitlines()
train_fr = Path(os.path.join(path, 'train.fr')).read_text(encoding='utf-8').splitlines()
val_en = Path(os.path.join(path, 'val.en')).read_text(encoding='utf-8').splitlines()
val_fr = Path(os.path.join(path, 'val.fr')).read_text(encoding='utf-8').splitlines()

train_df = pd.DataFrame({'en': train_en, 'fr': train_fr}).iloc[:10_000]
val_df = pd.DataFrame({'en': val_en,   'fr': val_fr  }).iloc[:1_000]

# Build vocab
vocab_en = Vocabulary(VOCAB_SIZE)
vocab_fr = Vocabulary(VOCAB_SIZE)
vocab_en.build(train_df['en'])
vocab_fr.build(train_df['fr'])

# Slicing gives:
#   dec_input  = [SOS, w1, ..., EOS, PAD]
#   dec_target = [w1,  w2, ..., EOS, PAD]
X_en = np.array([vocab_en.encode(s, MAX_LEN) for s in train_df['en']])
X_fr = np.array([vocab_fr.encode(s, MAX_LEN) for s in train_df['fr']])

dec_input  = X_fr[:, :-1]
dec_target = X_fr[:, 1:]

# Train test split
(X_en_tr, X_en_val, X_fr_in_tr, X_fr_in_val, X_fr_out_tr, X_fr_out_val) = train_test_split(X_en, dec_input, dec_target, test_size=0.1, random_state=42)

train_ds = TranslationDataset(np.hstack([X_en_tr, X_fr_in_tr]), X_fr_out_tr)
val_ds = TranslationDataset(np.hstack([X_en_val, X_fr_in_val]), X_fr_out_val)

# Keep src / tgt split sizes
SRC_LEN = X_en_tr.shape[1]
TGT_LEN = X_fr_in_tr.shape[1]


def collate(batch):
    combined, tgt = zip(*batch)
    combined = torch.stack(combined)
    src = combined[:, :SRC_LEN]
    dec_in = combined[:, SRC_LEN:]
    tgt = torch.stack(tgt)
    return src, dec_in, tgt

# prepare data
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_loader = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)


Using Colab cache for faster access to the 'multi30k-dataset' dataset.


In [63]:
# Build Model
encoder = Encoder(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)
decoder = Decoder(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)
model = Seq2Seq(encoder, decoder).to(device)

print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(10000, 256, padding_idx=0)
    (lstm): LSTM(256, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(10000, 256, padding_idx=0)
    (lstm): LSTM(256, 512, batch_first=True)
    (attention): ScaledDotProductAttention()
    (fc): Linear(in_features=1024, out_features=10000, bias=True)
  )
)


In [64]:
# Train Model
epochs = 5
criterion = nn.CrossEntropyLoss(ignore_index=0)   # ignore PAD
optimizer = optim.Adam(model.parameters())

for epoch in range(1, epochs + 1):
    # train
    model.train()
    total_loss = 0
    total_correct = 0
    total_tokens = 0

    for src, dec_in, tgt in train_loader:
        # Get updates and loss for model
        src, dec_in, tgt = src.to(device), dec_in.to(device), tgt.to(device)
        optimizer.zero_grad()
        logits = model(src, dec_in)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B * T, V), tgt.reshape(B * T))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * B

        # accuracy over non-pad tokens
        mask = tgt != 0
        preds = logits.argmax(dim=-1)
        total_correct += (preds == tgt)[mask].sum().item()
        total_tokens += mask.sum().item()

    # Final train loss and accuracy
    train_loss = total_loss / len(train_loader.dataset)
    train_acc = total_correct / total_tokens

    # validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_tokens = 0
    with torch.no_grad():
        for src, dec_in, tgt in val_loader:
            # Calculate validation loss and get correct/incorrect predictions
            src, dec_in, tgt = src.to(device), dec_in.to(device), tgt.to(device)
            logits = model(src, dec_in)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), tgt.reshape(B * T))
            val_loss += loss.item() * B
            mask = tgt != 0
            preds = logits.argmax(dim=-1)
            val_correct += (preds == tgt)[mask].sum().item()
            val_tokens += mask.sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc   = val_correct / val_tokens
    print(f'Epoch {epoch:2d}/{epochs}  loss={train_loss:.4f}  acc={train_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

Epoch  1/5  loss=5.0470  acc=0.2211  val_loss=4.1115  val_acc=0.2836
Epoch  2/5  loss=3.5526  acc=0.3569  val_loss=3.4324  val_acc=0.4169
Epoch  3/5  loss=2.6844  acc=0.4798  val_loss=2.9832  val_acc=0.5025
Epoch  4/5  loss=1.9334  acc=0.5804  val_loss=2.7472  val_acc=0.5504
Epoch  5/5  loss=1.2856  acc=0.7000  val_loss=2.6224  val_acc=0.5768


In [65]:
# translate function
def translate_sentence(model, src_sentence, vocab_en, vocab_fr, max_decode=40, device=device):
    model.eval()
    src_ids = vocab_en.encode(src_sentence, MAX_LEN)
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    with torch.no_grad():
        enc_out, h, c = model.encoder(src)
        # squeeze num_layers dim so states stay clean across steps
        h = h.squeeze(0)
        c = c.squeeze(0)
        dec_input = torch.tensor([[vocab_fr.SOS]], dtype=torch.long).to(device)
        pred_ids  = []

        for _ in range(max_decode):
            # Generate word by word
            logits, h, c = model.decoder(dec_input, enc_out,
                                         h.unsqueeze(0), c.unsqueeze(0))
            h = h.squeeze(0)
            c = c.squeeze(0)
            tok = logits[:, -1, :].argmax(dim=-1).item()
            if tok == vocab_fr.PAD or tok == vocab_fr.EOS:  # stop on EOS too
                break
            pred_ids.append(tok)
            dec_input = torch.tensor([[tok]], dtype=torch.long).to(device)

    return vocab_fr.decode(pred_ids)

In [66]:
# BLEU evaluation
smoothie = SmoothingFunction().method4
bleu_scores = []

for i in range(10):
    # Print some examples to compare the real french to the predicted french
    pred_tokens = translate_sentence(model, val_df['en'].iloc[i], vocab_en, vocab_fr)
    true_tokens = val_df['fr'].iloc[i].lower().split()

    print(f'Example {i+1}:')
    print(f"  English : {val_df['en'].iloc[i]}")
    print(f"  True FR : {' '.join(true_tokens)}")
    print(f"  Pred FR : {' '.join(pred_tokens)}\n")

    # Calculate bleu scores
    if pred_tokens:
        bleu_scores.append(sentence_bleu([true_tokens], pred_tokens, smoothing_function=smoothie))

print(f'Average BLEU Score: {np.mean(bleu_scores):.4f}')

Example 1:
  English : A group of men are loading cotton onto a truck
  True FR : un groupe d'hommes chargent du coton dans un camion
  Pred FR : un groupe d'hommes prennent un objet sur un terrain.

Example 2:
  English : A man sleeping in a green room on a couch.
  True FR : un homme dormant dans une chambre verte sur un canapé.
  Pred FR : un homme dormant dans un vert vert sur un trottoir.

Example 3:
  English : A boy wearing headphones sits on a woman's shoulders.
  True FR : un garçon avec un casque est assis sur les épaules d'une femme.
  Pred FR : un garçon portant un casque est assis sur un canapé en bois.

Example 4:
  English : Two men setting up a blue ice fishing hut on an iced over lake
  True FR : deux hommes installant une tente de pêche sur glace bleue sur un lac gelé
  Pred FR : deux hommes font une randonnée en bleu, bleue sur un tapis bleu sur un gril.

Example 5:
  English : A balding man wearing a red life jacket is sitting in a small boat.
  True FR : un homme c

## Part 4

In [67]:
# Transformer hyperparameters
D_MODEL   = 128
NUM_HEADS = 4
DFF       = 512
NUM_LAYERS = 2
DROPOUT   = 0.1
# VOCAB_SIZE, MAX_LEN already defined above (10000 size / 40 len)

# Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_length=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


# MultiHead Attention implementation
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.depth = d_model // num_heads
        self.d_model = d_model

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, T, _ = x.size()
        x = x.view(B, T, self.num_heads, self.depth)
        return x.transpose(1, 2)

    def forward(self, v, k, q, mask=None):
        B = q.size(0)
        q = self.split_heads(self.wq(q))
        k = self.split_heads(self.wk(k))
        v = self.split_heads(self.wv(v))

        d_k = self.depth ** 0.5
        scores = torch.matmul(q, k.transpose(-2, -1)) / d_k
        if mask is not None:
            scores = scores + mask

        weights = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(weights, v)

        out = out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.dense(out)


# Transformer Block (shared encoder / decoder)
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dff, dropout=0.1, is_decoder=False):
        super().__init__()
        self.is_decoder = is_decoder

        self.mha1 = MultiHeadAttention(d_model, num_heads, dropout)
        if is_decoder:
            self.mha2 = MultiHeadAttention(d_model, num_heads, dropout)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dff, d_model),
        )

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        if is_decoder:
            self.dropout3 = nn.Dropout(dropout)
            self.norm3 = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, enc_output=None, look_ahead_mask=None, padding_mask=None, enc_padding_mask=None):
        mask = look_ahead_mask if self.is_decoder else padding_mask
        attn1 = self.mha1(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn1))

        if self.is_decoder and enc_output is not None:
            attn2 = self.mha2(enc_output, enc_output, x, enc_padding_mask)
            x = self.norm2(x + self.dropout2(attn2))
            x = self.norm3(x + self.dropout3(self.ffn(x)))
        else:
            x = self.norm2(x + self.dropout2(self.ffn(x)))

        return x


# Full transformer
class SimplifiedTransformer(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                 num_heads=NUM_HEADS, dff=DFF, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model

        self.enc_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.dec_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, dropout)

        self.encoder_layers = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, dff, dropout, is_decoder=False) for _ in range(num_layers)]
        )
        self.decoder_layers = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, dff, dropout, is_decoder=True) for _ in range(num_layers)]
        )
        self.final_layer = nn.Linear(d_model, vocab_size)

    # Masks
    def make_padding_mask(self, seq):
        mask = (seq == 0).unsqueeze(1).unsqueeze(2).float()
        return mask * -1e9

    def make_look_ahead_mask(self, size, device):
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
        return mask * -1e9

    def forward(self, inp, tar):
            # Encoder part
            x = self.enc_embedding(inp) * math.sqrt(self.d_model)
            x = self.pos_encoding(x)
            enc_mask = self.make_padding_mask(inp)
            for layer in self.encoder_layers:
                x = layer(x, padding_mask=enc_mask)
            enc_output = x

            # Decoder part
            T_tar = tar.size(1)
            x = self.dec_embedding(tar) * math.sqrt(self.d_model)
            x = self.pos_encoding(x)

            # Create both masks (dimensions on size because mismatch for broadcasting had messed me up for hours)
            la_mask = self.make_look_ahead_mask(T_tar, tar.device)          # (T, T)
            dec_pad_mask = self.make_padding_mask(tar)                      # (B, 1, 1, T)

            la_mask = la_mask.unsqueeze(0).unsqueeze(0)                     # (1, 1, T, T)

            dec_pad_mask = dec_pad_mask.expand(-1, -1, T_tar, -1)           # (B, 1, T, T)

            combined = torch.minimum(dec_pad_mask, la_mask)

            for layer in self.decoder_layers:
                x = layer(x, enc_output=enc_output,
                          look_ahead_mask=combined,
                          enc_padding_mask=enc_mask)

            return self.final_layer(x)


In [68]:
# Define Model
transformer = SimplifiedTransformer().to(device)
print(transformer)

SimplifiedTransformer(
  (enc_embedding): Embedding(10000, 128, padding_idx=0)
  (dec_embedding): Embedding(10000, 128, padding_idx=0)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder_layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (mha1): MultiHeadAttention(
        (wq): Linear(in_features=128, out_features=128, bias=True)
        (wk): Linear(in_features=128, out_features=128, bias=True)
        (wv): Linear(in_features=128, out_features=128, bias=True)
        (dense): Linear(in_features=128, out_features=128, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
      )
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
      (norm1): LayerNo

In [69]:
# Need a new encoder with a start of sequence for the decoder
def encode_with_sos(sentence, vocab, max_len):
    ids = [vocab.word2idx.get(w, vocab.UNK) for w in sentence.lower().split()]
    ids = [vocab.SOS] + ids[:max_len - 2]
    ids.append(vocab.EOS)
    ids += [vocab.PAD] * (max_len - len(ids))
    return ids

X_en_t = np.array([vocab_en.encode(s, MAX_LEN) for s in train_df['en']])
X_fr_t = np.array([encode_with_sos(s, vocab_fr, MAX_LEN) for s in train_df['fr']])

dec_in_t = X_fr_t[:, :-1]
dec_tgt_t = X_fr_t[:, 1:]

# Setup data for the transformer again
X_en_tr_t, X_en_val_t, X_fr_in_tr_t, X_fr_in_val_t, X_fr_out_tr_t, X_fr_out_val_t = train_test_split(
    X_en_t, dec_in_t, dec_tgt_t, test_size=0.1, random_state=42)

transformer_epochs = 15
transformer_batch = 64

tr_en = torch.tensor(X_en_tr_t, dtype=torch.long)
tr_fr = torch.tensor(X_fr_in_tr_t, dtype=torch.long)
tr_tgt = torch.tensor(X_fr_out_tr_t, dtype=torch.long)
va_en = torch.tensor(X_en_val_t, dtype=torch.long)
va_fr = torch.tensor(X_fr_in_val_t, dtype=torch.long)
va_tgt = torch.tensor(X_fr_out_val_t, dtype=torch.long)

t_train_ds = torch.utils.data.TensorDataset(tr_en, tr_fr, tr_tgt)
t_val_ds = torch.utils.data.TensorDataset(va_en, va_fr, va_tgt)
t_train_loader = DataLoader(t_train_ds, batch_size=transformer_batch, shuffle=True)
t_val_loader = DataLoader(t_val_ds,   batch_size=transformer_batch, shuffle=False)

# Set label_smoothing=0.1 to prevent reliance on EOS early
t_criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

# Follow warmup schedule from "Attention is All You Need" so model does not collapse and predict only EOS
warmup_steps = 4000
d_model_val  = D_MODEL

def lr_lambda(step):
    step = max(step, 1)
    return (d_model_val ** -0.5) * min(step ** -0.5, step * warmup_steps ** -1.5)

t_optimizer = optim.Adam(transformer.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
t_scheduler = optim.lr_scheduler.LambdaLR(t_optimizer, lr_lambda)

# Training
for epoch in range(1, transformer_epochs + 1):
    transformer.train()
    total_loss, total_correct, total_tokens = 0, 0, 0

    for src, dec_in, tgt in t_train_loader:
        # Update weights per batch
        src, dec_in, tgt = src.to(device), dec_in.to(device), tgt.to(device)
        t_optimizer.zero_grad()
        logits = transformer(src, dec_in)
        B, T, V = logits.shape
        loss = t_criterion(logits.reshape(B * T, V), tgt.reshape(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer.parameters(), 1.0)
        t_optimizer.step()
        t_scheduler.step()   # step every batch, not every epoch

        # Get training loss and accuracy
        total_loss += loss.item() * B
        mask = tgt != 0
        total_correct += (logits.argmax(-1) == tgt)[mask].sum().item()
        total_tokens  += mask.sum().item()

    # Final aggregated loss and accuracy
    train_loss = total_loss / len(t_train_loader.dataset)
    train_acc  = total_correct / total_tokens

    transformer.eval()
    val_loss, val_correct, val_tokens = 0, 0, 0
    with torch.no_grad():
        for src, dec_in, tgt in t_val_loader:
            # Get vaidation loss and accuracy
            src, dec_in, tgt = src.to(device), dec_in.to(device), tgt.to(device)
            logits = transformer(src, dec_in)
            B, T, V = logits.shape
            loss = t_criterion(logits.reshape(B * T, V), tgt.reshape(B * T))
            val_loss += loss.item() * B
            mask = tgt != 0
            val_correct += (logits.argmax(-1) == tgt)[mask].sum().item()
            val_tokens  += mask.sum().item()

    # Final aggregated val loss and  accuracy
    val_loss /= len(t_val_loader.dataset)
    val_acc   = val_correct / val_tokens
    print(f'Epoch {epoch:2d}/{transformer_epochs}  loss={train_loss:.4f}  acc={train_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  lr={t_scheduler.get_last_lr()[0]:.6f}')


Epoch  1/15  loss=8.9858  acc=0.0423  val_loss=8.2760  val_acc=0.1128  lr=0.000049
Epoch  2/15  loss=7.6810  acc=0.1187  val_loss=6.9429  val_acc=0.1494  lr=0.000099
Epoch  3/15  loss=6.4226  acc=0.1609  val_loss=6.0035  val_acc=0.1835  lr=0.000148
Epoch  4/15  loss=5.7577  acc=0.2197  val_loss=5.4966  val_acc=0.2635  lr=0.000197
Epoch  5/15  loss=5.3308  acc=0.2728  val_loss=5.1385  val_acc=0.3028  lr=0.000246
Epoch  6/15  loss=5.0098  acc=0.3048  val_loss=4.8784  val_acc=0.3345  lr=0.000296
Epoch  7/15  loss=4.7575  acc=0.3324  val_loss=4.6746  val_acc=0.3561  lr=0.000345
Epoch  8/15  loss=4.5570  acc=0.3561  val_loss=4.5184  val_acc=0.3779  lr=0.000394
Epoch  9/15  loss=4.3843  acc=0.3773  val_loss=4.3934  val_acc=0.3945  lr=0.000443
Epoch 10/15  loss=4.2347  acc=0.3961  val_loss=4.2780  val_acc=0.4175  lr=0.000493
Epoch 11/15  loss=4.0954  acc=0.4148  val_loss=4.1853  val_acc=0.4345  lr=0.000542
Epoch 12/15  loss=3.9633  acc=0.4319  val_loss=4.1082  val_acc=0.4460  lr=0.000591
Epoc

In [70]:
# translate/inference
def translate_transformer(model, src_sentence, vocab_en, vocab_fr,
                           max_decode=40, device=device):
    model.eval()
    src_ids = vocab_en.encode(src_sentence, MAX_LEN)
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    dec_input = torch.tensor([[vocab_fr.SOS]], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_decode):
            logits = model(src, dec_input)
            next_tok = logits[:, -1, :].argmax(dim=-1).item()

            # Append token, then check stop conditions (doing other way around led to EOS)
            dec_input = torch.cat(
                [dec_input, torch.tensor([[next_tok]], dtype=torch.long).to(device)],
                dim=1
            )
            if next_tok == vocab_fr.EOS or next_tok == vocab_fr.PAD:
                break

    # decode everything after the leading SOS (decode() strips EOS/PAD)
    pred_ids = dec_input[0, 1:].tolist()
    return vocab_fr.decode(pred_ids)


# BLEU evaluation
smoothie_t = SmoothingFunction().method4
t_bleu_scores = []

for i in range(10):
    pred_tokens = translate_transformer(transformer, val_df['en'].iloc[i], vocab_en, vocab_fr)
    true_tokens = val_df['fr'].iloc[i].lower().split()

    print(f'Example {i+1}:')
    print(f"  English : {val_df['en'].iloc[i]}")
    print(f"  True FR : {' '.join(true_tokens)}")
    print(f"  Pred FR : {' '.join(pred_tokens) if pred_tokens else '<empty>'}")
    print()

    if pred_tokens:
        t_bleu_scores.append(sentence_bleu([true_tokens], pred_tokens, smoothing_function=smoothie_t))

if t_bleu_scores:
    print(f'Transformer Average BLEU Score: {np.mean(t_bleu_scores):.4f}')
else:
    print("No valid predictions - model may need more training epochs.")


Example 1:
  English : A group of men are loading cotton onto a truck
  True FR : un groupe d'hommes chargent du coton dans un camion
  Pred FR : un groupe d'hommes faisant une journée ensoleillée.

Example 2:
  English : A man sleeping in a green room on a couch.
  True FR : un homme dormant dans une chambre verte sur un canapé.
  Pred FR : un homme en vert fait une figure sur un banc de sable.

Example 3:
  English : A boy wearing headphones sits on a woman's shoulders.
  True FR : un garçon avec un casque est assis sur les épaules d'une femme.
  Pred FR : un garçon vêtu d'un t-shirt gris est assis sur une moto

Example 4:
  English : Two men setting up a blue ice fishing hut on an iced over lake
  True FR : deux hommes installant une tente de pêche sur glace bleue sur un lac gelé
  Pred FR : deux hommes sur un cheval bleu qui porte un terrain de la tête.

Example 5:
  English : A balding man wearing a red life jacket is sitting in a small boat.
  True FR : un homme chauve vêtu d'un 

The LSTM seq2seq model that we implemented first in part 2 performed better than the transformer model from part 4. The seq2seq model got a BLEU score of 0.2034 while the transformer model only got a score of 0.1596. This can likely be attributed to the slower learning rate of the transformer model. The seq2seq model started to overfit very rapidly and plateaued in validation accuracy around epoch 5. In comparison, the transformer model was still improving after 15 epochs. If given more epochs to train, it is likely that the transformer model would outperform the seq2seq model we made in part 2. Another distinction is that this transformer model is undersized and typically performs better on larger datasets. Comparatively, the seq2seq model was able to perform better on a smaller dataset like multi30k. Per epoch, the transformer model trained faster as attention can be parallel while the LSTM was sequential. However, the overall time to train took longer on the transformer to reach the approximate performance of the LSTM.